In [1]:
import numpy as np

In [2]:
# Basic Unary ufuncs list excluding sqrt
METHODS_LIST = ['abs','fabs','square','exp','log','log10','log2','log1p','sign','ceil','floor','rint','modf'
             ,'isnan','isfinite','isinf','cos','cosh','sin','sinh','tan','tanh','arccos','arccosh','arcsin'
             ,'arcsinh','arctan','arctanh','logical_not']

In [3]:
class B():
    def __init__(self, input_array, *index):
        if(len(index) == 0):
            self.idx = None
        elif(len(index) == 4):
            s1 = slice(index[0],index[1])
            s2 = slice(index[2],index[3])
            self.idx = (s1,s2)
        elif(type(index[0]) is str):
            self.idx = eval('np.s_'+index[0])
        else:
            self.idx = None
        self.arr = input_array
        self.mask = np.zeros(self.arr.shape, dtype=bool)
        self.mask[self.idx] = 1
    
    def sqrt(self):
    # At locations -where the condition is True, the -out array will be set to the ufunc result.
    # Where the condition is False, the -out array will be set to zero.
        return np.sqrt(np.abs(self.arr), where=self.mask, out=np.zeros(self.arr.shape))
    
    # convert the original 2D array into a 3D one by copying
    def repeat3d(self, repeats=2):
        return np.repeat(self.arr[:, :, np.newaxis], repeats, axis=2)

In [4]:
# Add ufunc methods to class B
for i in METHODS_LIST:
    # define function
    exec(f"def {i}(self):\
         return np.{i}(self.arr, where=self.mask, out=np.zeros(self.arr.shape))")
    
    #  set attribute to class B
    exec(f"setattr(B, '{i}', {i})")
    
    # delete function
    exec(f"del {i}")

In [5]:
# example #1
a = np.arange(25).reshape(5,5)**2
print(a)

[[  0   1   4   9  16]
 [ 25  36  49  64  81]
 [100 121 144 169 196]
 [225 256 289 324 361]
 [400 441 484 529 576]]


In [6]:
# without slice
obj = B(a)

In [7]:
print(obj.sqrt())

[[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]
 [15. 16. 17. 18. 19.]
 [20. 21. 22. 23. 24.]]


In [8]:
# with slice a,b,c,d
obj = B(a,1,4,1,4)

In [9]:
print(obj.sqrt())

[[ 0.  0.  0.  0.  0.]
 [ 0.  6.  7.  8.  0.]
 [ 0. 11. 12. 13.  0.]
 [ 0. 16. 17. 18.  0.]
 [ 0.  0.  0.  0.  0.]]


In [10]:
# with slice [a:b,c:d] as str
obj = B(a,'[1:4,1:4]')

In [11]:
print(obj.sqrt())

[[ 0.  0.  0.  0.  0.]
 [ 0.  6.  7.  8.  0.]
 [ 0. 11. 12. 13.  0.]
 [ 0. 16. 17. 18.  0.]
 [ 0.  0.  0.  0.  0.]]


In [12]:
# example #2
np.random.seed(42)
b = np.random.randn(16).reshape(4,4)
print(b)

[[ 0.49671415 -0.1382643   0.64768854  1.52302986]
 [-0.23415337 -0.23413696  1.57921282  0.76743473]
 [-0.46947439  0.54256004 -0.46341769 -0.46572975]
 [ 0.24196227 -1.91328024 -1.72491783 -0.56228753]]


In [13]:
obj = B(b,'[2:,1:]')

In [14]:
print(obj.fabs())

[[0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.54256004 0.46341769 0.46572975]
 [0.         1.91328024 1.72491783 0.56228753]]


In [15]:
print(obj.floor())

[[ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]
 [ 0.  0. -1. -1.]
 [ 0. -2. -2. -1.]]


In [16]:
# convert 2D into 3D
d = np.arange(9).reshape(3,3)
print(d)

[[0 1 2]
 [3 4 5]
 [6 7 8]]


In [17]:
obj = B(d)

In [18]:
d3d = obj.repeat3d(2)

In [19]:
print(d3d[:,:,0])
print(d3d[:,:,1])

[[0 1 2]
 [3 4 5]
 [6 7 8]]
[[0 1 2]
 [3 4 5]
 [6 7 8]]
